In [ ]:
import sys
import os
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("  Risk Management Module Initialized")

## 1. Generate Portfolio Data

In [ ]:
# Generate multi-asset portfolio returns
np.random.seed(42)
n_assets = 10
n_days = 1000
dates = pd.date_range(start='2020-01-01', periods=n_days, freq='D')

# Create correlation matrix
base_corr = 0.3
corr_matrix = np.full((n_assets, n_assets), base_corr)
np.fill_diagonal(corr_matrix, 1.0)

# Add some stronger correlations
for i in range(0, n_assets-1, 2):
    corr_matrix[i, i+1] = 0.7
    corr_matrix[i+1, i] = 0.7

# Generate returns with different volatilities
volatilities = np.linspace(0.01, 0.03, n_assets)  # Daily vol from 1% to 3%
means = np.linspace(0.0003, 0.0008, n_assets)  # Daily returns

# Cholesky decomposition for correlated returns
L = np.linalg.cholesky(corr_matrix)
uncorrelated = np.random.randn(n_days, n_assets)
correlated = uncorrelated @ L.T

# Scale by volatility and add drift
returns = correlated * volatilities + means

# Create DataFrame
asset_names = [f'Asset_{i+1}' for i in range(n_assets)]
returns_df = pd.DataFrame(returns, columns=asset_names, index=dates)

# Portfolio weights (not equal-weighted)
weights = np.array([0.15, 0.12, 0.10, 0.08, 0.08, 0.12, 0.10, 0.08, 0.09, 0.08])
weights = weights / weights.sum()  # Normalize

# Portfolio returns
portfolio_returns = (returns_df * weights).sum(axis=1)

# Calculate cumulative returns
cumulative_returns = (1 + returns_df).cumprod()
portfolio_cumulative = (1 + portfolio_returns).cumprod()

# Visualization
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Individual asset performance
for col in cumulative_returns.columns:
    axes[0].plot(cumulative_returns.index, cumulative_returns[col], alpha=0.5, linewidth=1)
axes[0].plot(portfolio_cumulative.index, portfolio_cumulative, 'r-', linewidth=2.5, label='Portfolio')
axes[0].set_ylabel('Cumulative Return')
axes[0].set_title('Asset and Portfolio Performance')
axes[0].legend(['Portfolio'], loc='upper left')
axes[0].grid(True, alpha=0.3)

# Portfolio returns distribution
axes[1].hist(portfolio_returns, bins=50, density=True, alpha=0.7, edgecolor='black')
mu, sigma = portfolio_returns.mean(), portfolio_returns.std()
x = np.linspace(portfolio_returns.min(), portfolio_returns.max(), 100)
axes[1].plot(x, stats.norm.pdf(x, mu, sigma), 'r-', linewidth=2, label='Normal Distribution')
axes[1].axvline(x=mu, color='green', linestyle='--', linewidth=2, label=f'Mean: {mu*100:.3f}%')
axes[1].set_xlabel('Daily Return')
axes[1].set_ylabel('Density')
axes[1].set_title('Portfolio Returns Distribution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Correlation heatmap
sns.heatmap(returns_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, 
           square=True, ax=axes[2], cbar_kws={'label': 'Correlation'})
axes[2].set_title('Asset Correlation Matrix')

plt.tight_layout()
plt.show()

print(f"Portfolio of {n_assets} assets over {n_days} days")
print(f"\nPortfolio Statistics:")
print(f"  Mean daily return: {portfolio_returns.mean()*100:.4f}%")
print(f"  Daily volatility: {portfolio_returns.std()*100:.4f}%")
print(f"  Annualized return: {portfolio_returns.mean()*252*100:.2f}%")
print(f"  Annualized volatility: {portfolio_returns.std()*np.sqrt(252)*100:.2f}%")
print(f"  Sharpe ratio: {(portfolio_returns.mean()/portfolio_returns.std())*np.sqrt(252):.2f}")
print(f"\nWeights: {dict(zip(asset_names, weights.round(3)))}")

## 2. Value at Risk (VaR) - Multiple Methods

In [ ]:
def historical_var(returns: np.ndarray, confidence_level: float = 0.95) -> float:
    """Calculate VaR using historical simulation"""
    return np.percentile(returns, (1 - confidence_level) * 100)

def parametric_var(returns: np.ndarray, confidence_level: float = 0.95) -> float:
    """Calculate VaR assuming normal distribution"""
    mu = returns.mean()
    sigma = returns.std()
    z_score = stats.norm.ppf(1 - confidence_level)
    return mu + z_score * sigma

def monte_carlo_var(returns: np.ndarray, confidence_level: float = 0.95, 
                   n_simulations: int = 10000) -> float:
    """Calculate VaR using Monte Carlo simulation"""
    mu = returns.mean()
    sigma = returns.std()
    simulated_returns = np.random.normal(mu, sigma, n_simulations)
    return np.percentile(simulated_returns, (1 - confidence_level) * 100)

def cornish_fisher_var(returns: np.ndarray, confidence_level: float = 0.95) -> float:
    """Calculate VaR using Cornish-Fisher expansion (accounts for skewness and kurtosis)"""
    mu = returns.mean()
    sigma = returns.std()
    skew = stats.skew(returns)
    kurt = stats.kurtosis(returns)
    
    # Cornish-Fisher quantile adjustment
    z = stats.norm.ppf(1 - confidence_level)
    z_cf = z + (z**2 - 1) * skew / 6 + (z**3 - 3*z) * kurt / 24 - (2*z**3 - 5*z) * skew**2 / 36
    
    return mu + z_cf * sigma

# Calculate VaR using different methods
confidence_levels = [0.90, 0.95, 0.99]
var_results = {}

for cl in confidence_levels:
    var_results[f'{cl:.0%}'] = {
        'Historical': historical_var(portfolio_returns, cl),
        'Parametric': parametric_var(portfolio_returns, cl),
        'Monte Carlo': monte_carlo_var(portfolio_returns, cl, 10000),
        'Cornish-Fisher': cornish_fisher_var(portfolio_returns, cl)
    }

var_df = pd.DataFrame(var_results).T

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# VaR comparison across methods
var_df.plot(kind='bar', ax=axes[0, 0], alpha=0.8, edgecolor='black')
axes[0, 0].set_ylabel('VaR (Daily Return)')
axes[0, 0].set_title('Value at Risk - Method Comparison')
axes[0, 0].set_xlabel('Confidence Level')
axes[0, 0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0, 0].grid(True, alpha=0.3, axis='y')
axes[0, 0].legend(title='Method')

# Distribution with VaR levels (95%)
axes[0, 1].hist(portfolio_returns, bins=50, density=True, alpha=0.7, edgecolor='black')
for method, var_value in var_results['95%'].items():
    axes[0, 1].axvline(x=var_value, linestyle='--', linewidth=2, label=f'{method}: {var_value*100:.3f}%')
axes[0, 1].set_xlabel('Daily Return')
axes[0, 1].set_ylabel('Density')
axes[0, 1].set_title('95% VaR - All Methods')
axes[0, 1].legend(fontsize=8)
axes[0, 1].grid(True, alpha=0.3)

# VaR in dollar terms (assuming $1M portfolio)
portfolio_value = 1_000_000
var_dollars = var_df * portfolio_value

var_dollars.plot(kind='bar', ax=axes[1, 0], alpha=0.8, edgecolor='black')
axes[1, 0].set_ylabel('VaR ($)')
axes[1, 0].set_title(f'Value at Risk in Dollars (${portfolio_value:,} Portfolio)')
axes[1, 0].set_xlabel('Confidence Level')
axes[1, 0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1, 0].grid(True, alpha=0.3, axis='y')
axes[1, 0].legend(title='Method')

# Q-Q plot (check normality assumption)
stats.probplot(portfolio_returns, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot - Test for Normality')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== Value at Risk Analysis ===")
print("\nVaR as percentage:")
print(var_df.round(6) * 100)
print(f"\nVaR in dollars (${portfolio_value:,} portfolio):")
print(var_dollars.round(2))

# Normality test
statistic, p_value = stats.jarque_bera(portfolio_returns)
print(f"\nJarque-Bera test for normality:")
print(f"  Statistic: {statistic:.4f}")
print(f"  P-value: {p_value:.4f}")
print(f"  Normal distribution: {'No' if p_value < 0.05 else 'Yes'} (at 5% significance)")

## 3. Conditional Value at Risk (CVaR / Expected Shortfall)

In [ ]:
def calculate_cvar(returns: np.ndarray, confidence_level: float = 0.95) -> float:
    """Calculate CVaR (Expected Shortfall) - average loss beyond VaR"""
    var = historical_var(returns, confidence_level)
    # Average of returns worse than VaR
    cvar = returns[returns <= var].mean()
    return cvar

# Calculate CVaR for different confidence levels
cvar_results = {}
for cl in confidence_levels:
    var = historical_var(portfolio_returns, cl)
    cvar = calculate_cvar(portfolio_returns, cl)
    cvar_results[f'{cl:.0%}'] = {'VaR': var, 'CVaR': cvar, 'Difference': cvar - var}

cvar_df = pd.DataFrame(cvar_results).T

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# VaR vs CVaR comparison
x = np.arange(len(confidence_levels))
width = 0.35

axes[0, 0].bar(x - width/2, cvar_df['VaR'] * 100, width, label='VaR', alpha=0.8, edgecolor='black')
axes[0, 0].bar(x + width/2, cvar_df['CVaR'] * 100, width, label='CVaR', alpha=0.8, edgecolor='black')
axes[0, 0].set_ylabel('Risk Measure (%)')
axes[0, 0].set_title('VaR vs CVaR Comparison')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels([f'{cl:.0%}' for cl in confidence_levels])
axes[0, 0].set_xlabel('Confidence Level')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3, axis='y')

# Distribution showing VaR and CVaR (95%)
var_95 = historical_var(portfolio_returns, 0.95)
cvar_95 = calculate_cvar(portfolio_returns, 0.95)
tail_losses = portfolio_returns[portfolio_returns <= var_95]

axes[0, 1].hist(portfolio_returns, bins=50, density=True, alpha=0.5, edgecolor='black', label='All Returns')
axes[0, 1].hist(tail_losses, bins=20, density=True, alpha=0.8, color='red', 
               edgecolor='black', label='Tail Losses')
axes[0, 1].axvline(x=var_95, color='orange', linestyle='--', linewidth=2, label=f'VaR: {var_95*100:.3f}%')
axes[0, 1].axvline(x=cvar_95, color='red', linestyle='--', linewidth=2, label=f'CVaR: {cvar_95*100:.3f}%')
axes[0, 1].set_xlabel('Daily Return')
axes[0, 1].set_ylabel('Density')
axes[0, 1].set_title('95% VaR and CVaR - Tail Distribution')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# CVaR ratio (CVaR/VaR) - measures tail risk
cvar_ratio = np.abs(cvar_df['CVaR'] / cvar_df['VaR'])
axes[1, 0].bar(range(len(cvar_ratio)), cvar_ratio, alpha=0.8, color='purple', edgecolor='black')
axes[1, 0].axhline(y=1, color='red', linestyle='--', linewidth=2, label='Ratio = 1')
axes[1, 0].set_ylabel('CVaR/VaR Ratio')
axes[1, 0].set_title('Tail Risk Ratio (Higher = Fatter Tails)')
axes[1, 0].set_xticks(range(len(confidence_levels)))
axes[1, 0].set_xticklabels([f'{cl:.0%}' for cl in confidence_levels])
axes[1, 0].set_xlabel('Confidence Level')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(cvar_ratio):
    axes[1, 0].text(i, v, f'{v:.3f}', ha='center', va='bottom', fontweight='bold')

# Dollar values
cvar_dollars = cvar_df[['VaR', 'CVaR']] * portfolio_value
cvar_dollars.plot(kind='bar', ax=axes[1, 1], alpha=0.8, edgecolor='black')
axes[1, 1].set_ylabel('Amount ($)')
axes[1, 1].set_title(f'Risk Measures in Dollars (${portfolio_value:,} Portfolio)')
axes[1, 1].set_xlabel('Confidence Level')
axes[1, 1].grid(True, alpha=0.3, axis='y')
axes[1, 1].legend(['VaR', 'CVaR'])

plt.tight_layout()
plt.show()

print("\n=== Conditional Value at Risk (CVaR) Analysis ===")
print("\nRisk measures as percentage:")
print(cvar_df.round(6) * 100)
print(f"\nRisk measures in dollars (${portfolio_value:,} portfolio):")
print(cvar_dollars.round(2))
print(f"\nTail risk ratios (CVaR/VaR):")
print(cvar_ratio.round(3))

## 4. Portfolio Risk Decomposition

In [ ]:
# Calculate portfolio risk metrics
cov_matrix = returns_df.cov() * 252  # Annualized
portfolio_variance = weights @ cov_matrix @ weights
portfolio_volatility = np.sqrt(portfolio_variance)

# Marginal contribution to risk (MCR)
mcr = cov_matrix @ weights / portfolio_volatility

# Component contribution to risk (CCR)
ccr = weights * mcr

# Percentage contribution
pct_contribution = ccr / portfolio_volatility * 100

# Create results DataFrame
risk_decomp = pd.DataFrame({
    'Weight': weights,
    'Volatility': volatilities * np.sqrt(252),  # Annualized
    'MCR': mcr,
    'CCR': ccr,
    'Contribution (%)': pct_contribution
}, index=asset_names)

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Weight vs risk contribution
x = np.arange(len(asset_names))
width = 0.35

axes[0, 0].bar(x - width/2, risk_decomp['Weight'] * 100, width, 
              label='Weight', alpha=0.8, edgecolor='black')
axes[0, 0].bar(x + width/2, risk_decomp['Contribution (%)'], width, 
              label='Risk Contribution', alpha=0.8, edgecolor='black')
axes[0, 0].set_ylabel('Percentage (%)')
axes[0, 0].set_title('Portfolio Weight vs Risk Contribution')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(asset_names, rotation=45, ha='right')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3, axis='y')

# Risk contribution breakdown
colors = plt.cm.tab10(np.linspace(0, 1, len(asset_names)))
axes[0, 1].pie(risk_decomp['Contribution (%)'], labels=asset_names, autopct='%1.1f%%',
              colors=colors, startangle=90)
axes[0, 1].set_title('Risk Contribution Breakdown')

# Scatter: weight vs volatility, sized by risk contribution
scatter = axes[1, 0].scatter(risk_decomp['Weight'] * 100, 
                            risk_decomp['Volatility'] * 100,
                            s=risk_decomp['Contribution (%)'] * 100,
                            alpha=0.6, c=range(len(asset_names)), cmap='viridis',
                            edgecolors='black', linewidths=2)
axes[1, 0].set_xlabel('Portfolio Weight (%)')
axes[1, 0].set_ylabel('Asset Volatility (%)')
axes[1, 0].set_title('Weight vs Volatility (Size = Risk Contribution)')
axes[1, 0].grid(True, alpha=0.3)
for i, asset in enumerate(asset_names):
    axes[1, 0].annotate(asset, 
                       (risk_decomp['Weight'].iloc[i] * 100, 
                        risk_decomp['Volatility'].iloc[i] * 100),
                       fontsize=8, alpha=0.7)

# Component contribution to risk (CCR)
axes[1, 1].barh(asset_names, risk_decomp['CCR'] * 100, alpha=0.8, edgecolor='black')
axes[1, 1].set_xlabel('Component Contribution to Risk (%)')
axes[1, 1].set_title('Absolute Risk Contribution by Asset')
axes[1, 1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\n=== Portfolio Risk Decomposition ===")
print(f"\nPortfolio volatility: {portfolio_volatility*100:.2f}%")
print(f"\nRisk decomposition by asset:")
print(risk_decomp.round(4))
print(f"\nValidation - sum of contributions: {risk_decomp['Contribution (%)'].sum():.2f}%")

# Identify top risk contributors
top_contributors = risk_decomp.nlargest(3, 'Contribution (%)')
print(f"\nTop 3 risk contributors:")
for idx, row in top_contributors.iterrows():
    print(f"  {idx}: {row['Contribution (%)']:.2f}% (weight: {row['Weight']*100:.2f}%)")

## 5. Stress Testing & Scenario Analysis

In [ ]:
def stress_test_scenario(returns_df, weights, scenario_name, return_shocks):
    """Apply return shocks to assets and calculate portfolio impact"""
    shocked_returns = return_shocks
    portfolio_shock = (shocked_returns * weights).sum()
    
    return {
        'Scenario': scenario_name,
        'Portfolio Impact (%)': portfolio_shock * 100,
        'Dollar Impact': portfolio_shock * portfolio_value
    }

# Define stress scenarios
scenarios = {
    'Market Crash (-20%)': np.array([-0.20] * n_assets),
    'Sector Rotation': np.array([-0.15, -0.15, -0.15, -0.15, -0.15, 0.10, 0.10, 0.10, 0.10, 0.10]),
    'Volatility Spike': np.array([-0.10, -0.12, -0.08, -0.15, -0.11, -0.09, -0.13, -0.10, -0.08, -0.14]),
    'Flight to Quality': np.array([-0.05, -0.05, -0.05, -0.05, -0.05, 0.03, 0.03, 0.03, 0.03, 0.03]),
    'Correlation Breakdown': np.array([-0.25, 0.05, -0.20, 0.08, -0.15, 0.10, -0.18, 0.06, -0.12, 0.04]),
    'Gradual Decline (-5%)': np.array([-0.05] * n_assets),
}

# Run stress tests
stress_results = []
for scenario_name, shocks in scenarios.items():
    result = stress_test_scenario(returns_df, weights, scenario_name, shocks)
    stress_results.append(result)

stress_df = pd.DataFrame(stress_results).set_index('Scenario')

# Historical stress test - find worst periods
rolling_returns = portfolio_returns.rolling(window=21).sum()  # 1-month
worst_periods = rolling_returns.nsmallest(5)

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Stress test results
stress_df['Portfolio Impact (%)'].plot(kind='barh', ax=axes[0, 0], color='red', 
                                       alpha=0.7, edgecolor='black')
axes[0, 0].axvline(x=0, color='black', linestyle='-', linewidth=1)
axes[0, 0].set_xlabel('Portfolio Impact (%)')
axes[0, 0].set_title('Stress Test Scenarios - Portfolio Impact')
axes[0, 0].grid(True, alpha=0.3, axis='x')

# Dollar impact
stress_df['Dollar Impact'].plot(kind='barh', ax=axes[0, 1], color='darkred', 
                               alpha=0.7, edgecolor='black')
axes[0, 1].axvline(x=0, color='black', linestyle='-', linewidth=1)
axes[0, 1].set_xlabel('Dollar Impact ($)')
axes[0, 1].set_title(f'Stress Test - Dollar Impact (${portfolio_value:,} Portfolio)')
axes[0, 1].grid(True, alpha=0.3, axis='x')

# Historical worst periods
axes[1, 0].bar(range(len(worst_periods)), worst_periods.values * 100, 
              alpha=0.7, color='orange', edgecolor='black')
axes[1, 0].set_xlabel('Period Rank')
axes[1, 0].set_ylabel('21-Day Return (%)')
axes[1, 0].set_title('Top 5 Worst Historical Periods (21-day rolling)')
axes[1, 0].set_xticks(range(len(worst_periods)))
axes[1, 0].set_xticklabels([f'{i+1}' for i in range(len(worst_periods))])
axes[1, 0].grid(True, alpha=0.3, axis='y')
for i, (date, ret) in enumerate(worst_periods.items()):
    axes[1, 0].text(i, ret * 100, f"{date.strftime('%Y-%m-%d')}", 
                   ha='center', va='top', fontsize=7, rotation=45)

# Monte Carlo stress test
n_scenarios = 10000
mc_shocks = np.random.normal(-0.05, 0.08, (n_scenarios, n_assets))  # Mean -5%, std 8%
mc_portfolio_impacts = (mc_shocks * weights).sum(axis=1)

axes[1, 1].hist(mc_portfolio_impacts * 100, bins=50, density=True, 
               alpha=0.7, color='purple', edgecolor='black')
axes[1, 1].axvline(x=np.percentile(mc_portfolio_impacts, 5) * 100, 
                  color='red', linestyle='--', linewidth=2, label='5th Percentile')
axes[1, 1].axvline(x=np.mean(mc_portfolio_impacts) * 100, 
                  color='green', linestyle='--', linewidth=2, label='Mean')
axes[1, 1].set_xlabel('Portfolio Impact (%)')
axes[1, 1].set_ylabel('Density')
axes[1, 1].set_title('Monte Carlo Stress Test (10,000 scenarios)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== Stress Testing Results ===")
print("\nPredefined scenarios:")
print(stress_df.round(2))

print("\n5 Worst historical periods (21-day returns):")
for date, ret in worst_periods.items():
    print(f"  {date.strftime('%Y-%m-%d')}: {ret*100:.2f}%")

print("\nMonte Carlo stress test summary:")
print(f"  Mean impact: {np.mean(mc_portfolio_impacts)*100:.2f}%")
print(f"  5th percentile: {np.percentile(mc_portfolio_impacts, 5)*100:.2f}%")
print(f"  1st percentile: {np.percentile(mc_portfolio_impacts, 1)*100:.2f}%")
print(f"  Probability of >10% loss: {(mc_portfolio_impacts < -0.10).sum() / n_scenarios * 100:.2f}%")

## 6. Maximum Drawdown Analysis

In [ ]:
def calculate_drawdowns(cumulative_returns):
    """Calculate drawdown series"""
    running_max = cumulative_returns.expanding().max()
    drawdown = (cumulative_returns - running_max) / running_max
    return drawdown

def find_drawdown_periods(cumulative_returns, drawdowns):
    """Identify distinct drawdown periods"""
    in_drawdown = drawdowns < 0
    drawdown_periods = []
    
    start = None
    for i, (date, in_dd) in enumerate(in_drawdown.items()):
        if in_dd and start is None:
            start = i
        elif not in_dd and start is not None:
            end = i - 1
            period_dd = drawdowns.iloc[start:end+1]
            max_dd = period_dd.min()
            max_dd_date = period_dd.idxmin()
            
            drawdown_periods.append({
                'start': drawdowns.index[start],
                'end': drawdowns.index[end],
                'peak_date': max_dd_date,
                'max_drawdown': max_dd,
                'duration_days': end - start + 1
            })
            start = None
    
    return sorted(drawdown_periods, key=lambda x: x['max_drawdown'])[:5]  # Top 5

# Calculate drawdowns
drawdowns = calculate_drawdowns(portfolio_cumulative)
max_drawdown = drawdowns.min()
max_dd_date = drawdowns.idxmin()

# Find drawdown periods
dd_periods = find_drawdown_periods(portfolio_cumulative, drawdowns)

# Recovery analysis
current_dd = drawdowns.iloc[-1]
days_since_peak = (drawdowns.index[-1] - portfolio_cumulative.expanding().idxmax().iloc[-1]).days

# Visualization
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Portfolio value and drawdowns
axes[0].plot(portfolio_cumulative.index, portfolio_cumulative, 'b-', linewidth=2)
axes[0].scatter([max_dd_date], [portfolio_cumulative.loc[max_dd_date]], 
               color='red', s=200, zorder=5, marker='v', label='Max Drawdown')
axes[0].set_ylabel('Cumulative Return')
axes[0].set_title('Portfolio Cumulative Returns')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Drawdown chart
axes[1].fill_between(drawdowns.index, 0, drawdowns * 100, color='red', alpha=0.3)
axes[1].plot(drawdowns.index, drawdowns * 100, 'r-', linewidth=1.5)
axes[1].axhline(y=max_drawdown * 100, color='darkred', linestyle='--', 
               linewidth=2, label=f'Max DD: {max_drawdown*100:.2f}%')
axes[1].set_ylabel('Drawdown (%)')
axes[1].set_title('Drawdown Over Time')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Underwater plot with recovery periods
axes[2].fill_between(drawdowns.index, 0, drawdowns * 100, 
                     where=(drawdowns < 0), color='red', alpha=0.5, label='Underwater')
axes[2].fill_between(drawdowns.index, 0, drawdowns * 100, 
                     where=(drawdowns >= 0), color='green', alpha=0.5, label='New High')
axes[2].set_xlabel('Date')
axes[2].set_ylabel('Drawdown (%)')
axes[2].set_title('Underwater Plot')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Additional drawdown statistics
print("\n=== Maximum Drawdown Analysis ===")
print(f"\nMaximum drawdown: {max_drawdown*100:.2f}%")
print(f"Date of maximum drawdown: {max_dd_date.strftime('%Y-%m-%d')}")
print(f"Current drawdown: {current_dd*100:.2f}%")
print(f"Days since last peak: {days_since_peak}")

print("\nTop 5 drawdown periods:")
for i, period in enumerate(dd_periods, 1):
    print(f"\n{i}. {period['start'].strftime('%Y-%m-%d')} to {period['end'].strftime('%Y-%m-%d')}")
    print(f"   Max drawdown: {period['max_drawdown']*100:.2f}%")
    print(f"   Duration: {period['duration_days']} days")
    print(f"   Peak date: {period['peak_date'].strftime('%Y-%m-%d')}")

# Calmar ratio (annualized return / max drawdown)
annualized_return = portfolio_returns.mean() * 252
calmar_ratio = annualized_return / abs(max_drawdown)
print(f"\nCalmar Ratio: {calmar_ratio:.2f}")

## 7. Summary

### Key Risk Metrics:

1. **Value at Risk (VaR)**
   - Historical, parametric, Monte Carlo methods
   - Confidence levels: 90%, 95%, 99%
   - Limitations: doesn't capture tail risk

2. **Conditional VaR (CVaR)**
   - Expected shortfall beyond VaR
   - Better measure of tail risk
   - Coherent risk measure

3. **Risk Decomposition**
   - Marginal contribution to risk
   - Component contributions
   - Identifies risk concentrations

4. **Stress Testing**
   - Scenario analysis
   - Historical worst periods
   - Monte Carlo simulations

5. **Maximum Drawdown**
   - Peak-to-trough decline
   - Recovery periods
   - Calmar ratio

### Best Practices:
- Use multiple risk measures (VaR + CVaR)
- Regular stress testing
- Monitor risk decomposition
- Set position and portfolio limits
- Account for correlation changes
- Test extreme scenarios
- Track drawdowns in real-time